In [ ]:
# ==============================
# 1. Importy
# ==============================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.2f}'.format)

# Powtarzalność wyników
np.random.seed(42)
torch.manual_seed(42)

# ==============================
# 2. Tworzenie przykładowych danych
# ==============================
# Dane syntetyczne:
# powierzchnia_m2, liczba_pokoi, wiek_budynku -> cena
n_samples = 1000

powierzchnia = np.random.uniform(30, 200, n_samples)
pokoje = np.random.randint(1, 8, n_samples)
wiek = np.random.randint(0, 50, n_samples)

# Tworzymy zależność z lekkim szumem
cena = (
    powierzchnia * 9500
    + pokoje * 18000
    - wiek * 2000
    + np.random.normal(0, 30000, n_samples)
)

df = pd.DataFrame({
    "powierzchnia_m2": powierzchnia,
    "liczba_pokoi": pokoje,
    "wiek_budynku": wiek,
    "cena": cena
})

print(df.head())

# ==============================
# 3. Przygotowanie danych
# ==============================
X = df[["powierzchnia_m2", "liczba_pokoi", "wiek_budynku"]].values
y = df["cena"].values.reshape(-1, 1)

# Podział na train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Skalowanie cech wejściowych
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Skalowanie targetu też pomaga w regresji
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

# Zamiana na tensory PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# ==============================
# 4. Definicja modelu
# ==============================
class HousePriceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

model = HousePriceModel()
print(model)

# ==============================
# 5. Loss i optimizer
# ==============================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ==============================
# 6. Trening
# ==============================
epochs = 100
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

# ==============================
# 7. Wykres loss
# ==============================
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Loss w trakcie treningu")
plt.xlabel("Epoka")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

# ==============================
# 8. Ewaluacja na zbiorze testowym
# ==============================
model.eval()
with torch.no_grad():
    test_predictions_scaled = model(X_test_tensor)
    test_loss = criterion(test_predictions_scaled, y_test_tensor).item()

print(f"Test MSE (na danych zeskalowanych): {test_loss:.4f}")

# Cofamy skalowanie do normalnych wartości
test_predictions = scaler_y.inverse_transform(test_predictions_scaled.numpy())
y_test_original = scaler_y.inverse_transform(y_test_tensor.numpy())

# Liczymy prosty MAE
mae = np.mean(np.abs(test_predictions - y_test_original))
print(f"Średni błąd bezwzględny (MAE): {mae:.2f} PLN")

# ==============================
# 9. Przykładowe predykcje
# ==============================
results = pd.DataFrame({
    "Rzeczywista cena": y_test_original.flatten()[:20],
    "Przewidziana cena": test_predictions.flatten()[:20]
})

print(results)

# ==============================
# 10. Predykcja dla nowego przykładu
# ==============================
# przykład:
# dom 120 m2, 4 pokoje, 10 lat
new_house = np.array([[40, 2, 0]])

new_house_scaled = scaler_X.transform(new_house)
new_house_tensor = torch.tensor(new_house_scaled, dtype=torch.float32)

model.eval()
with torch.no_grad():
    predicted_price_scaled = model(new_house_tensor).numpy()

predicted_price = scaler_y.inverse_transform(predicted_price_scaled)

print(f"\nPrzewidywana cena dla nowego domu[40m2 / 2pokoje / NOWY]: {predicted_price[0][0]:.2f} PLN")